# Week 3 In-Class Exercise: Classifying Fashion Images

In the textbook (Chapter 3), we built classifiers for the **MNIST** dataset of handwritten digits. We measured performance with cross-validation, confusion matrices, precision/recall, the precision-recall trade-off, and ROC curves, then moved on to multiclass classification.

In this exercise, you'll apply those **same concepts** to a different but structurally identical dataset:

> **Fashion-MNIST** — 70,000 grayscale 28x28 images of clothing items across 10 categories (T-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot).

Because Fashion-MNIST has the exact same shape as MNIST (784 features, 10 classes), every technique from the textbook notebook transfers directly. Your job is to adapt the code, not invent new methods.

You'll follow the same workflow as Chapter 3:
1. Load and explore the data
2. Train a **binary** classifier ("is it a sneaker?")
3. Measure performance: cross-validation accuracy, confusion matrix, precision/recall/F1
4. Explore the **precision/recall trade-off** using decision thresholds
5. Plot an **ROC curve** and compare two classifiers
6. Build a **multiclass** classifier and analyze its errors

**Data source:** [Fashion-MNIST on OpenML](https://www.openml.org/d/40996) (Zalando Research).

### Useful API References

| Task | Documentation |
|------|---------------|
| Load data | [sklearn.datasets.fetch_openml](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html) |
| SGD classifier | [sklearn.linear_model.SGDClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) |
| Cross-validation | [sklearn.model_selection.cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) / [cross_val_predict](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html) |
| Confusion matrix | [sklearn.metrics.confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) |
| Precision / Recall / F1 | [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html), [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html), [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) |
| PR curve | [precision_recall_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html) |
| ROC curve | [roc_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html), [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html) |
| Random forest | [sklearn.ensemble.RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) |
| Scaling | [sklearn.preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) |

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week3/week3_inclass_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## Step 1: Load the Data

Just like the textbook loads MNIST with `fetch_openml('mnist_784', ...)`, we load Fashion-MNIST. This cell is provided for you — just run it. (The first run downloads ~50 MB and may take a minute.)

In [ ]:
from sklearn.datasets import fetch_openml

fashion = fetch_openml('Fashion-MNIST', as_frame=False)
X, y = fashion.data, fashion.target

# Human-readable names for the 10 classes (label '0'..'9')
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print('X shape:', X.shape)   # (70000, 784)
print('y shape:', y.shape)   # (70000,)
print('Labels: ', np.unique(y))

### Peek at an image

This helper mirrors `plot_digit()` from the textbook. Each row of `X` is a flattened 28x28 image. Run this to see a few examples.

In [ ]:
def plot_image(image_data):
    image = image_data.reshape(28, 28)
    plt.imshow(image, cmap='binary')
    plt.axis('off')

plt.figure(figsize=(9, 2))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plot_image(X[i])
    plt.title(class_names[int(y[i])], fontsize=11)
plt.show()

### Train/test split

Fashion-MNIST (like MNIST) is already shuffled and arranged so the first 60,000 rows are the training set and the last 10,000 are the test set. Run this cell.

In [ ]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]
print('Train:', X_train.shape, '  Test:', X_test.shape)

## Step 2: Train a Binary Classifier ("Is it a Sneaker?")

In the textbook, the first classifier answered a yes/no question: *"is this a 5?"* We'll do the same, but our question is *"is this a **sneaker**?"* (class label `'7'`).

**Your turn!** First, build the boolean target vectors. Then train an `SGDClassifier`.

**Hint (from the textbook):**
```python
y_train_5 = (y_train == '5')
y_test_5 = (y_test == '5')

from sklearn.linear_model import SGDClassifier
sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_5)
```

In [ ]:
# The class label for 'Sneaker' is '7'
SNEAKER = '7'

# TODO: Create boolean targets y_train_sneaker and y_test_sneaker
# y_train_sneaker = ...
# y_test_sneaker  = ...

# TODO: Import SGDClassifier, create it with random_state=42, and fit it
#       on X_train and y_train_sneaker.
# Your code here:



# TODO: Predict whether the first training image (X[0]) is a sneaker.
# Hint: sgd_clf.predict([X[0]])


## Step 3: Measure Accuracy with Cross-Validation

**Your turn!** Use 3-fold cross-validation to estimate accuracy, exactly like the textbook.

**Hint:**
```python
from sklearn.model_selection import cross_val_score
cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring='accuracy')
```

In [ ]:
# TODO: Compute 3-fold cross-validation accuracy for sgd_clf on the sneaker task.
# Your code here:




**Question:** The accuracy is probably around 97-98%. But only 1 in 10 images is a sneaker. If a "dumb" classifier always guessed *"not a sneaker,"* what accuracy would it get? Why does this make accuracy a misleading metric here?

*Your answer:*



## Step 4: Confusion Matrix, Precision, Recall, and F1

Because accuracy is misleading for imbalanced data, we look at the confusion matrix and the precision/recall metrics.

**Your turn!** First get cross-validated predictions with `cross_val_predict`, then build the confusion matrix and compute precision, recall, and F1.

**Hint (from the textbook):**
```python
from sklearn.model_selection import cross_val_predict
y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)

from sklearn.metrics import confusion_matrix
confusion_matrix(y_train_5, y_train_pred)

from sklearn.metrics import precision_score, recall_score, f1_score
precision_score(y_train_5, y_train_pred)
recall_score(y_train_5, y_train_pred)
f1_score(y_train_5, y_train_pred)
```

In [ ]:
# TODO: Get cross-validated predictions (y_train_pred) with cross_val_predict.
# Your code here:



# TODO: Print the confusion matrix.


# TODO: Print precision, recall, and F1 score.



**Question:** In the confusion matrix, the four cells are [[TN, FP], [FN, TP]]. In plain English, what does a **false positive** mean for the sneaker task, and what does a **false negative** mean?

*Your answer:*



## Step 5: The Precision/Recall Trade-off

The classifier makes decisions by comparing a **score** to a threshold. Raising the threshold increases precision but lowers recall (and vice versa). The textbook visualizes this with a precision-recall-vs-threshold plot.

**Your turn!** Get decision scores via `cross_val_predict` with `method='decision_function'`, then compute the precision-recall curve.

**Hint (from the textbook):**
```python
y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3,
                             method='decision_function')
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)
```

In [ ]:
# TODO: Compute y_scores using cross_val_predict with method='decision_function'.
# Your code here:



# TODO: Compute precisions, recalls, thresholds with precision_recall_curve.



In [ ]:
# TODO: Plot precision and recall as functions of the threshold.
# Hint (from the textbook):
#   plt.plot(thresholds, precisions[:-1], 'b--', label='Precision')
#   plt.plot(thresholds, recalls[:-1], 'g-', label='Recall')
#   plt.xlabel('Threshold'); plt.legend(); plt.grid(); plt.show()
# Your code here:




**Your turn!** Suppose you want **at least 90% precision**. Find the lowest threshold that achieves it, and report the recall you'd get at that threshold.

**Hint (from the textbook):**
```python
idx_for_90 = (precisions >= 0.90).argmax()
threshold_for_90 = thresholds[idx_for_90]
y_train_pred_90 = (y_scores >= threshold_for_90)
```

In [ ]:
# TODO: Find the threshold for >=90% precision and report the resulting recall.
# Your code here:




## Step 6: The ROC Curve and Comparing Classifiers

The ROC curve plots the true positive rate against the false positive rate. The area under it (AUC) is a common single-number summary.

**Your turn!** Compute and plot the ROC curve for the SGD classifier, and print its AUC.

**Hint (from the textbook):**
```python
from sklearn.metrics import roc_curve, roc_auc_score
fpr, tpr, thresholds = roc_curve(y_train_5, y_scores)
roc_auc_score(y_train_5, y_scores)
```

In [ ]:
# TODO: Compute fpr, tpr, thresholds with roc_curve and plot the ROC curve.
#       Also plot the diagonal 'no-skill' line: plt.plot([0, 1], [0, 1], 'k:')
# TODO: Print the AUC with roc_auc_score.
# Your code here:




**Your turn!** Now train a `RandomForestClassifier` and compare it to the SGD classifier. A random forest exposes `predict_proba` instead of `decision_function`, so use the probability of the positive class as the score.

**Hint (from the textbook):**
```python
from sklearn.ensemble import RandomForestClassifier
forest_clf = RandomForestClassifier(random_state=42)
y_probas_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3,
                                    method='predict_proba')
y_scores_forest = y_probas_forest[:, 1]   # probability of the positive class
```
Then compute `roc_auc_score(y_train_sneaker, y_scores_forest)` and compare the two AUCs. (This cell may take a couple of minutes.)

In [ ]:
# TODO: Train a RandomForestClassifier via cross_val_predict(method='predict_proba'),
#       take the positive-class probability as the score, and print its AUC.
# TODO: Print both AUCs side by side. Which classifier is better?
# Your code here:




**Question:** Which model had the higher ROC AUC? Give one reason a random forest often outperforms a linear model like SGD on image pixels.

*Your answer:*



## Step 7: Multiclass Classification

Now classify all 10 clothing categories at once (not just sneaker vs. not-sneaker).

**Your turn!** Train an `SGDClassifier` on the full `y_train` (all 10 labels) and estimate its accuracy with 3-fold cross-validation. To keep runtime reasonable, you may train on a subset (e.g. the first 10,000 rows).

**Hint (from the textbook):**
```python
sgd_clf.fit(X_train, y_train)          # scikit-learn handles OvR automatically
cross_val_score(sgd_clf, X_train, y_train, cv=3, scoring='accuracy')
```

In [ ]:
# Optional: use a subset to keep things fast
X_sub, y_sub = X_train[:10000], y_train[:10000]

# TODO: Fit an SGDClassifier on the full-label data and report cross-val accuracy.
# Your code here:




**Your turn!** The textbook shows that **scaling the inputs** boosts SGD accuracy. Scale the features with `StandardScaler` and re-run cross-validation.

**Hint (from the textbook):**
```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sub.astype('float64'))
cross_val_score(sgd_clf, X_scaled, y_sub, cv=3, scoring='accuracy')
```

In [ ]:
# TODO: Scale the features and re-run cross-validation. Did accuracy improve?
# Your code here:




## Step 8: Error Analysis

The textbook uses a normalized confusion matrix to see *which classes get confused with which*.

**Your turn!** Get cross-validated predictions on the scaled data, then display a confusion matrix normalized by row.

**Hint (from the textbook):**
```python
from sklearn.metrics import ConfusionMatrixDisplay
y_sub_pred = cross_val_predict(sgd_clf, X_scaled, y_sub, cv=3)
ConfusionMatrixDisplay.from_predictions(y_sub, y_sub_pred,
                                        display_labels=class_names,
                                        normalize='true', values_format='.0%',
                                        xticks_rotation=90)
plt.show()
```

In [ ]:
# TODO: Build and display a row-normalized confusion matrix for the 10 classes.
# Your code here:




**Question:** Which pairs of clothing categories get confused most often? Does that make intuitive sense given what the images look like? (Think about how similar a shirt, a coat, and a pullover look at 28x28.)

*Your answer:*



## Reflection Questions

Answer each question in the cell below it (1-3 sentences each).

**Q1:** Why is accuracy a poor metric for the binary "is it a sneaker?" task, and what should you look at instead?

*Your answer:*



**Q2:** Describe a real-world clothing-classification scenario where you would prefer **high precision**, and one where you would prefer **high recall**.

*Your answer:*



**Q3:** ROC AUC and the precision-recall curve both summarize a classifier across thresholds. When is the precision-recall curve more informative than the ROC curve?

*Your answer:*



**Q4:** The confusion matrix revealed specific categories that are hard to tell apart. Name two things you could try to reduce those errors (features, model choice, preprocessing, or data).

*Your answer:*

